# 02. Preprocessing and Feature Engineering

Builds the modelling table:

1. merge `admissions` + `patients` + `diagnoses_icd` + `d_icd_diagnoses`
2. compute length of stay, gap between visits, and the 30/60-day readmission flags
3. count repeat visits per (patient, diagnosis) and derive `previous_admissions_count` / `visit_order`
4. restrict to kidney-related diagnoses
5. score diagnosis severity from the ICD long title with a rule-based keyword lexicon

Output: `data/processed/kidney_cohort.parquet`


In [2]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

RAW = Path("../data/raw/hosp")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)


def raw(name: str) -> Path:
    """Locate a MIMIC-IV table, accepting either .csv or .csv.gz."""
    for ext in (".csv.gz", ".csv"):
        path = RAW / f"{name}{ext}"
        if path.exists():
            return path
    raise FileNotFoundError(
        f"{name}.csv or {name}.csv.gz not found in {RAW.resolve()}. "
        "See the README for the expected data/raw/hosp layout."
    )


## 1. Load and merge

In [3]:
admissions = pd.read_csv(
    raw("admissions"),
    usecols=["subject_id", "hadm_id", "admittime", "dischtime", "admission_type",
             "admission_location", "insurance", "hospital_expire_flag"],
    parse_dates=["admittime", "dischtime"],
)
patients = pd.read_csv(raw("patients"), usecols=["subject_id", "gender", "anchor_age"])
diagnoses = pd.read_csv(
    raw("diagnoses_icd"),
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"],
)
d_icd = pd.read_csv(
    raw("d_icd_diagnoses"),
    usecols=["icd_code", "icd_version", "long_title"],
)

diagnoses = diagnoses.merge(d_icd, on=["icd_code", "icd_version"], how="left")
df = admissions.merge(patients, on="subject_id", how="left")

print(f"admission-level rows: {len(df):,}")

admission-level rows: 431,231


## 2. Readmission indicators

Computed at the **admission** level, before the diagnosis join, so that one hospital stay
contributes one gap and one label.

- `los_hours`: discharge minus admission, in hours
- `time_diff`: current admission time minus the *previous* discharge time for the same patient
- `readmission_under_30_days` / `readmission_under_60_days`: whether that gap is under the threshold


In [4]:
df = df.sort_values(["subject_id", "admittime"]).reset_index(drop=True)

df["los_hours"] = (df["dischtime"] - df["admittime"]).dt.total_seconds() / 3600
df["los_hours"] = df["los_hours"].round(2)

prev_disch = df.groupby("subject_id")["dischtime"].shift(1)
df["time_diff"] = df["admittime"] - prev_disch
gap_days = df["time_diff"].dt.total_seconds() / 86400

df["readmission_under_30_days"] = ((gap_days >= 0) & (gap_days <= 30)).astype(int)
df["readmission_under_60_days"] = ((gap_days >= 0) & (gap_days <= 60)).astype(int)

df[["subject_id", "hadm_id", "admittime", "dischtime", "los_hours",
    "time_diff", "readmission_under_30_days"]].head(8)

,subject_id,hadm_id,admittime,dischtime,los_hours,time_diff,readmission_under_30_days
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,18.87,NaT,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,24.37,50 days 01:12:00,0
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,53.33,25 days 17:46:00,1
3,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,42.10,11 days 05:49:00,1
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,7.17,NaT,0
5,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00,108.93,NaT,0
6,10000084,29888819,2160-12-28 05:11:00,2160-12-28 16:07:00,10.93,32 days 14:19:00,0
7,10000108,27250926,2163-09-27 23:17:00,2163-09-28 09:04:00,9.78,NaT,0


In [5]:
rate = df["readmission_under_30_days"].mean()
print(f"30-day readmission rate (admission level): {rate:.1%}")
print(f"60-day readmission rate (admission level): {df['readmission_under_60_days'].mean():.1%}")

30-day readmission rate (admission level): 20.2%
60-day readmission rate (admission level): 26.7%


## 3. Visit history features

- `visit_order`: the sequence number of the stay for that patient. MIMIC-IV dates are shifted,
  so ordering by `admittime` within a patient is safe even though the absolute dates are not real.
- `previous_admissions_count`: number of stays that came before the current one.


In [6]:
df["visit_order"] = df.groupby("subject_id").cumcount() + 1
df["previous_admissions_count"] = df["visit_order"] - 1

df[["subject_id", "hadm_id", "visit_order", "previous_admissions_count"]].head(8)

,subject_id,hadm_id,visit_order,previous_admissions_count
0,10000032,22595853,1,0
1,10000032,22841357,2,1
2,10000032,29079034,3,2
3,10000032,25742920,4,3
4,10000068,25022803,1,0
5,10000084,23052089,1,0
6,10000084,29888819,2,1
7,10000108,27250926,1,0


## 4. Attach diagnoses and count repeat visits

`readmit` is the number of separate admissions on which a patient carried the same ICD code.
This is the "recurrent readmissions by subject_id and icd_code" table from the original project.


In [7]:
df = df.merge(diagnoses, on=["subject_id", "hadm_id"], how="inner")
print(f"admission x diagnosis rows: {len(df):,}")

repeat = (
    df.groupby(["subject_id", "icd_code"])["hadm_id"]
    .nunique()
    .reset_index(name="readmit")
)
df = df.merge(repeat, on=["subject_id", "icd_code"], how="left")

repeat[repeat["readmit"] > 1].head(10)

admission x diagnosis rows: 4,756,326


,subject_id,icd_code,readmit
4,10000032,2761,3
5,10000032,2767,2
7,10000032,29680,2
8,10000032,3051,3
11,10000032,496,4
12,10000032,5715,4
15,10000032,78959,4
17,10000032,V08,3
19,10000032,V462,2
22,10000084,E785,2


## 5. Kidney cohort

The original analysis found acute kidney failure among the top diagnoses driving readmissions,
then widened the scope from that single code to any kidney-related diagnosis. Same logic here:
match on the ICD long title.


In [8]:
KIDNEY_PATTERN = re.compile(
    r"kidney|renal|nephr|dialysis|glomerul|nephrotic|end stage renal",
    flags=re.IGNORECASE,
)

df["long_title"] = df["long_title"].fillna("")
kidney = df[df["long_title"].str.contains(KIDNEY_PATTERN)].copy()

print(f"kidney rows     : {len(kidney):,}")
print(f"unique patients : {kidney['subject_id'].nunique():,}")
print(f"unique stays    : {kidney['hadm_id'].nunique():,}")
print(f"positive rate   : {kidney['readmission_under_30_days'].mean():.1%}")

kidney["long_title"].value_counts().head(12)

kidney rows     : 261,347
unique patients : 47,919
unique stays    : 111,891
positive rate   : 25.8%


long_title
Acute kidney failure, unspecified                                                                                                                             47856
Chronic kidney disease, unspecified                                                                                                                           26354
Hypertensive chronic kidney disease, unspecified, with chronic kidney disease stage I through stage IV, or unspecified                                        22715
End stage renal disease                                                                                                                                       14016
Hypertensive chronic kidney disease with stage 1 through stage 4 chronic kidney disease, or unspecified chronic kidney disease                                10178
Type 2 diabetes mellitus with diabetic chronic kidney disease                                                                                                 10003
Anemi

## 6. Rule-based severity scoring

A keyword lexicon maps diagnosis text to a 1-5 severity score. The lookup walks the lexicon in
descending severity and returns on the first hit, so the most serious matching phrase wins.
Anything unmatched falls back to 1 (least severe).


In [9]:
SEVERITY_KEYWORDS = {
    5: [
        "end stage renal disease", "end-stage renal", "stage 5", "stage v",
        "dialysis", "kidney transplant", "renal transplant",
    ],
    4: [
        "stage 4", "stage iv", "severe", "acute kidney failure",
        "acute renal failure", "necrosis", "nephrectomy",
    ],
    3: [
        "stage 3", "stage iii", "chronic kidney disease", "chronic renal",
        "renal insufficiency", "moderate", "nephritis", "nephropathy",
    ],
    2: [
        "stage 2", "stage ii", "hypertensive", "anemia", "mild",
        "renal osteodystrophy", "unspecified",
    ],
    1: ["stage 1", "stage i", "screening", "history of"],
}


def get_severity(long_title: str) -> int:
    """Return a 1-5 severity score for a diagnosis description.

    Walks the lexicon from most to least severe and returns the first match,
    defaulting to 1 when nothing matches.
    """
    text = str(long_title).lower()
    for score in sorted(SEVERITY_KEYWORDS, reverse=True):
        for keyword in SEVERITY_KEYWORDS[score]:
            if keyword in text:
                return score
    return 1


kidney["severity"] = kidney["long_title"].apply(get_severity)

kidney[["subject_id", "hadm_id", "icd_code", "long_title", "severity"]].head(10)

,subject_id,hadm_id,icd_code,long_title,severity
83,10000560,28979390,1890,"Malignant neoplasm of kidney, except pelvis",1
96,10000764,27897940,5849,"Acute kidney failure, unspecified",4
233,10000980,29654838,40390,"Hypertensive chronic kidney disease, unspecifi...",4
239,10000980,29654838,5853,"Chronic kidney disease, Stage III (moderate)",3
245,10000980,26913865,5854,"Chronic kidney disease, Stage IV (severe)",4
249,10000980,26913865,40390,"Hypertensive chronic kidney disease, unspecifi...",4
258,10000980,24947999,5854,"Chronic kidney disease, Stage IV (severe)",4
260,10000980,24947999,28521,Anemia in chronic kidney disease,3
263,10000980,24947999,40390,"Hypertensive chronic kidney disease, unspecifi...",4
280,10000980,25242409,5854,"Chronic kidney disease, Stage IV (severe)",4


In [10]:
kidney["severity"].value_counts().sort_index()

severity
1     33185
2     10125
3     72990
4    101065
5     43982
Name: count, dtype: int64

## 7. Save

In [11]:
out = PROCESSED / "kidney_cohort.parquet"
kidney.to_parquet(out, index=False)
print(f"saved {len(kidney):,} rows to {out}")

saved 261,347 rows to ..\data\processed\kidney_cohort.parquet
